In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
DRIVE_ROOT = "/content/drive/MyDrive/AthletiQ"
ZIP_PATH = os.path.join(DRIVE_ROOT, "soccernet_ball_yolo.zip")
MODEL_PATH = os.path.join(
    DRIVE_ROOT,
    "ball_detector_yolo11n_5ep",
    "weights",
    "last.pt"
)

for path in [ZIP_PATH, MODEL_PATH]:
    print(f"{'FOUND' if os.path.isfile(path) else 'MISSING'}: {path}")
    if os.path.isfile(path):
        print(f"  Size: {os.path.getsize(path) / (1024**3):.2f} GB")

FOUND: /content/drive/MyDrive/AthletiQ/soccernet_ball_yolo.zip
  Size: 17.05 GB
FOUND: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/last.pt
  Size: 0.01 GB


In [ ]:
import os
import shutil

RUNTIME_ZIP = "/content/soccernet_ball_yolo.zip"
RUNTIME_MODEL = "/content/last.pt"

print("Copying dataset ZIP...")
shutil.copy2(ZIP_PATH, RUNTIME_ZIP)
print(f"ZIP copied: {os.path.getsize(RUNTIME_ZIP) / (1024**3):.2f} GB")

print("\nCopying last checkpoint...")
shutil.copy2(MODEL_PATH, RUNTIME_MODEL)
print(f"Model copied: {os.path.getsize(RUNTIME_MODEL) / (1024**2):.2f} MB")

Copying dataset ZIP...
ZIP copied: 17.05 GB

Copying last checkpoint...
Model copied: 5.27 MB


In [ ]:
import os
import zipfile
import time

EXTRACT_DIR = "/content/soccernet_ball_yolo"
os.makedirs(EXTRACT_DIR, exist_ok=True)

print("Extracting dataset ZIP...")
start = time.time()

with zipfile.ZipFile(RUNTIME_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print(f"Extraction completed in {(time.time() - start) / 60:.2f} minutes.")

print("\nTop-level contents:")
for item in os.listdir(EXTRACT_DIR):
    print(" -", item)

Extracting dataset ZIP...
Extraction completed in 3.47 minutes.

Top-level contents:
 - soccernet_ball_yolo


In [ ]:
import os

matches = []

for current_root, dirs, files in os.walk(EXTRACT_DIR):
    if all(os.path.isdir(os.path.join(current_root, "images", split))
           for split in ["train", "val", "test"]):
        if all(os.path.isdir(os.path.join(current_root, "labels", split))
               for split in ["train", "val", "test"]):
            matches.append(current_root)

if len(matches) != 1:
    raise RuntimeError(f"Expected one dataset root; found: {matches}")

DATA_ROOT = matches[0]

print("Dataset root:", DATA_ROOT)

for split in ["train", "val", "test"]:
    image_dir = os.path.join(DATA_ROOT, "images", split)
    label_dir = os.path.join(DATA_ROOT, "labels", split)

    n_images = sum(
        f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
        for f in os.listdir(image_dir)
    )
    n_labels = sum(f.endswith(".txt") for f in os.listdir(label_dir))

    print(f"{split.upper()}: {n_images:,} images | {n_labels:,} labels")

Dataset root: /content/soccernet_ball_yolo/soccernet_ball_yolo
TRAIN: 55,500 images | 55,500 labels
VAL: 11,250 images | 11,250 labels
TEST: 12,750 images | 12,750 labels


In [ ]:
import yaml
import os

DATA_YAML = "/content/soccernet_ball.yaml"

data_config = {
    "path": DATA_ROOT,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "ball"}
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(open(DATA_YAML).read())

path: /content/soccernet_ball_yolo/soccernet_ball_yolo
train: images/train
val: images/val
test: images/test
names:
  0: ball



In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 11.9 MB/s eta 0:00:00


In [ ]:
import torch
from ultralytics import YOLO
import os

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available. Switch Colab to a GPU runtime.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

MODEL_PATH = "/content/last.pt"

model = YOLO(MODEL_PATH)

print("\nCheckpoint loaded:", MODEL_PATH)
print("Classes:", model.names)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
GPU: Tesla T4
VRAM: 14.56 GB

Checkpoint loaded: /content/last.pt
Classes: {0: 'ball'}


In [ ]:
from ultralytics import YOLO
import torch
import os
import json
from datetime import datetime

MODEL_PATH = "/content/last.pt"
DATA_YAML = "/content/soccernet_ball.yaml"

model = YOLO(MODEL_PATH)

test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=1280,
    batch=22,
    device=0,
    workers=4,
    plots=True,
    project="/content/ball_testing",
    name="yolo11n_last_test",
    exist_ok=True,
    verbose=True
)

metrics = {
    "checkpoint": "last.pt",
    "split": "test",
    "imgsz": 1280,
    "precision": float(test_results.box.mp),
    "recall": float(test_results.box.mr),
    "mAP50": float(test_results.box.map50),
    "mAP50_95": float(test_results.box.map),
    "evaluated_at": datetime.now().isoformat(timespec="seconds")
}

print("\n" + "=" * 55)
print("YOLO11n LAST CHECKPOINT — TEST RESULTS")
print("=" * 55)

for key, value in metrics.items():
    if isinstance(value, float) and key != "evaluated_at":
        print(f"{key:15s}: {value:.4f} ({value * 100:.2f}%)")
    else:
        print(f"{key:15s}: {value}")

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 68.0±35.3 MB/s, size: 263.0 KB)
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/test... 12750 images, 1240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12750/12750 606.4it/s 21.0s
val: New cache created: /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 580/580 1.7it/s 5:35
                   all      12750      11844        0.7      0.593       0.59      0.233
Speed: 3.1ms preprocess, 10.4ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/ball_testing/yolo11n_last_test

YOLO11n LAST CHECKPOINT — TEST RESULTS
checkpoint     : last.pt
split          : test
imgsz          : 1280
precision      : 0.6995 (6

In [ ]:
import os
import json
import shutil
import csv

DRIVE_TEST_DIR = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/last_checkpoint_test"
)
os.makedirs(DRIVE_TEST_DIR, exist_ok=True)

# Save metrics
json_path = os.path.join(DRIVE_TEST_DIR, "test_metrics.json")
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=4)

csv_path = os.path.join(DRIVE_TEST_DIR, "test_metrics.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=metrics.keys())
    writer.writeheader()
    writer.writerow(metrics)

# Save evaluation plots and supporting files, if generated
PLOT_DIR = "/content/ball_testing/yolo11n_last_test"

if os.path.isdir(PLOT_DIR):
    for filename in os.listdir(PLOT_DIR):
        source = os.path.join(PLOT_DIR, filename)

        # Copy plots and CSV/text evaluation artifacts, not redundant checkpoints
        if os.path.isfile(source) and filename.lower().endswith(
            (".png", ".jpg", ".csv", ".json", ".txt")
        ):
            shutil.copy2(source, os.path.join(DRIVE_TEST_DIR, filename))

print("Test results saved to:")
print(DRIVE_TEST_DIR)

for filename in sorted(os.listdir(DRIVE_TEST_DIR)):
    print(" -", filename)

Test results saved to:
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/last_checkpoint_test
 - BoxF1_curve.png
 - BoxPR_curve.png
 - BoxP_curve.png
 - BoxR_curve.png
 - confusion_matrix.png
 - confusion_matrix_normalized.png
 - test_metrics.csv
 - test_metrics.json
 - val_batch0_labels.jpg
 - val_batch0_pred.jpg
 - val_batch1_labels.jpg
 - val_batch1_pred.jpg
 - val_batch2_labels.jpg
 - val_batch2_pred.jpg


In [ ]:
import os
import pandas as pd

CSV_PATH = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/results.csv"
)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Training history not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

columns = [
    c for c in df.columns
    if any(k in c.lower() for k in [
        "epoch", "metrics/", "val/", "train/box",
        "train/cls", "train/dfl", "lr/"
    ])
]

print(df[columns].to_string(index=False))

 epoch  train/box_loss  train/cls_loss  train/dfl_loss  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss   lr/pg0   lr/pg1   lr/pg2
     1         2.57008         1.60182         0.82219               0.74274            0.56854           0.56707              0.22108       2.25814       1.75610       0.75768 0.000167 0.000167 0.066847
     2         2.41237         1.44646         0.79614               0.71101            0.62176           0.60508              0.24841       2.19272       1.93782       0.74396 0.000267 0.000267 0.033614
     3         2.31748         1.35327         0.78927               0.73413            0.63812           0.61922              0.24820       2.20139       1.61546       0.74452 0.000302 0.000302 0.000315
     4         2.14869         1.24671         0.81283               0.73233            0.63680           0.62549              0.24846       2.19027       1.43939       0.74476 0.00020

In [ ]:
from ultralytics import YOLO
import torch
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------
MODEL_PATH = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

DATA_YAML = "/content/soccernet_ball.yaml"

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Checkpoint missing: {MODEL_PATH}")

if not os.path.isfile(DATA_YAML):
    raise FileNotFoundError(f"Dataset YAML missing: {DATA_YAML}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

# --------------------------------------------------
# Initialize from the best Stage 1 checkpoint
# --------------------------------------------------
model = YOLO(MODEL_PATH)

print("Loaded checkpoint:", MODEL_PATH)
print("Classes:", model.names)

# --------------------------------------------------
# Stage 2 fine-tuning
# --------------------------------------------------
results = model.train(
    data=DATA_YAML,

    # Training budget
    epochs=5,
    imgsz=1280,
    batch=22,
    device=0,
    workers=4,

    # Optimizer and learning-rate schedule
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.05,
    cos_lr=True,
    warmup_epochs=1,

    # Regularization
    weight_decay=0.0005,

    # Tiny-ball augmentation
    mosaic=0.5,
    close_mosaic=3,
    mixup=0.0,
    copy_paste=0.0,

    degrees=0.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,

    # Early stopping and checkpoints
    patience=4,
    save=True,
    save_period=1,
    val=True,

    # Reproducibility
    seed=0,
    deterministic=True,

    # New output directory: do not overwrite Stage 1
    project="/content/ball_training",
    name="yolo11n_soccernet_stage2",
    exist_ok=True,

    verbose=True,
)

print("Stage 2 training completed.")
print("Inspect:")
print("/content/ball_training/yolo11n_soccernet_stage2/results.csv")
print("/content/ball_training/yolo11n_soccernet_stage2/weights/best.pt")

GPU: Tesla T4
VRAM: 14.56 GB
Loaded checkpoint: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt
Classes: {0: 'ball'}
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=22, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=3, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/soccernet_ball.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=1280, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.0

In [ ]:
from pathlib import Path
import shutil

SOURCE = Path(
    "/content/ball_training/yolo11n_soccernet_stage2"
)

DEST = Path(
    "/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2"
)

if not SOURCE.exists():
    raise FileNotFoundError(
        f"Stage 2 results not found at: {SOURCE}\n"
        "Check the actual training run directory before copying."
    )

DEST.mkdir(parents=True, exist_ok=True)

for item in SOURCE.iterdir():
    target = DEST / item.name

    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

print("Stage 2 results copied to Google Drive!\n")
print(f"Destination: {DEST}\n")

for path in sorted(DEST.rglob("*")):
    if path.is_file():
        print(
            f"{path.relative_to(DEST)} | "
            f"{path.stat().st_size / (1024**2):.2f} MB"
        )

Stage 2 results copied to Google Drive!

Destination: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2

BoxF1_curve.png | 0.10 MB
BoxPR_curve.png | 0.09 MB
BoxP_curve.png | 0.10 MB
BoxR_curve.png | 0.09 MB
args.yaml | 0.00 MB
confusion_matrix.png | 0.10 MB
confusion_matrix_normalized.png | 0.09 MB
labels.jpg | 0.09 MB
results.csv | 0.00 MB
results.png | 0.29 MB
train_batch0.jpg | 0.53 MB
train_batch1.jpg | 0.47 MB
train_batch2.jpg | 0.54 MB
train_batch5046.jpg | 0.50 MB
train_batch5047.jpg | 0.42 MB
train_batch5048.jpg | 0.46 MB
val_batch0_labels.jpg | 0.46 MB
val_batch0_pred.jpg | 0.47 MB
val_batch1_labels.jpg | 0.40 MB
val_batch1_pred.jpg | 0.40 MB
val_batch2_labels.jpg | 0.40 MB
val_batch2_pred.jpg | 0.40 MB
weights/best.pt | 5.27 MB
weights/epoch0.pt | 20.33 MB
weights/epoch1.pt | 20.33 MB
weights/epoch2.pt | 20.34 MB
weights/epoch3.pt | 20.34 MB
weights/epoch4.pt | 20.34 MB
weights/last.pt | 5.27 MB
